# 📈 Automação de Análise de Ações da Bolsa

## 🎯 Objetivo
Este notebook automatiza o processo de:
1. **Buscar** dados de ações da bolsa de valores usando a API do Yahoo Finance
2. **Analisar** os dados dos últimos 6 meses (máxima, mínima, cotação atual e variação)
3. **Visualizar** o histórico de preços em um gráfico
4. **Enviar** automaticamente um e-mail com as análises para o gestor

## 🔧 Tecnologias Utilizadas
- `yfinance` — coleta de dados financeiros
- `pandas` — manipulação de dados
- `matplotlib` — geração de gráficos
- `pyautogui` + `pyperclip` — automação do envio de e-mail

---
> **Autor:** Thiago de Mattos  
> **Versão:** 2.0

## 📦 1. Instalação das Bibliotecas

Execute esta célula apenas na **primeira vez** que rodar o notebook.  
As bibliotecas serão instaladas automaticamente no seu ambiente.

In [ ]:
# Instalação das dependências necessárias
# Execute esta célula apenas uma vez
!pip install yfinance pyautogui pyperclip matplotlib pandas --quiet
print("✅ Bibliotecas instaladas com sucesso!")

## 📚 2. Importação das Bibliotecas

In [ ]:
# Bibliotecas para dados financeiros e análise
import yfinance
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Bibliotecas para automação de e-mail
import pyautogui
import pyperclip
import time

print("✅ Todas as bibliotecas importadas com sucesso!")

## ⚙️ 3. Configurações Gerais

Altere as variáveis abaixo conforme necessário antes de executar o notebook.

In [ ]:
# ============================================================
# CONFIGURAÇÕES — altere aqui conforme necessário
# ============================================================

# E-mail do destinatário
EMAIL_DESTINATARIO = "gestor@empresa.com"

# Período de análise (opções: '1mo', '3mo', '6mo', '1y', '2y')
PERIODO_ANALISE = "6mo"

# Pausa entre ações do PyAutoGUI (em segundos)
PYAUTOGUI_PAUSE = 2

# ============================================================
print(f"✅ Configurações carregadas!")
print(f"   → Destinatário: {EMAIL_DESTINATARIO}")
print(f"   → Período de análise: {PERIODO_ANALISE}")

## 📊 4. Buscar Dados da Ação

Informe o código da ação no formato correto:
- **Bovespa (B3):** `PETR4.SA`, `VALE3.SA`, `ITUB4.SA`
- **NYSE/NASDAQ:** `AAPL`, `GOOGL`, `TSLA`

In [ ]:
def buscar_dados_acao(codigo, periodo):
    """
    Busca os dados históricos de uma ação.

    Parâmetros:
        codigo  (str): Código da ação (ex: 'PETR4.SA')
        periodo (str): Período de análise (ex: '6mo')

    Retorna:
        DataFrame com os dados históricos ou None em caso de erro.
    """
    try:
        print(f"🔍 Buscando dados de '{codigo}' para os últimos {periodo}...")
        ticker = yfinance.Ticker(codigo)
        dados = ticker.history(periodo)

        # Verifica se os dados foram encontrados
        if dados.empty:
            print(f"⚠️  Nenhum dado encontrado para '{codigo}'.")
            print("     Verifique se o código está correto (ex: PETR4.SA, VALE3.SA).")
            return None

        print(f"✅ Dados carregados com sucesso! ({len(dados)} registros encontrados)")
        return dados

    except Exception as e:
        print(f"❌ Erro ao buscar dados: {e}")
        return None


# Solicita o código da ação ao usuário
codigo = input("Digite o código da ação (ex: PETR4.SA): ").strip().upper()

# Busca os dados
dados = buscar_dados_acao(codigo, PERIODO_ANALISE)

if dados is not None:
    print(f"\n📋 Prévia dos dados:")
    display(dados.tail(5))

## 📉 5. Gerar Análises

Calcula os indicadores principais: **máxima**, **mínima**, **cotação atual** e **variação no período**.

In [ ]:
def gerar_analises(dados, codigo):
    """
    Calcula os principais indicadores financeiros da ação.

    Parâmetros:
        dados  (DataFrame): Dados históricos da ação
        codigo (str): Código da ação

    Retorna:
        dict com os indicadores calculados.
    """
    try:
        fechamento = dados['Close']

        maxima  = round(fechamento.max(), 2)
        minima  = round(fechamento.min(), 2)
        atual   = round(fechamento.iloc[-1], 2)
        inicial = round(fechamento.iloc[0], 2)

        # Calcula a variação percentual no período
        variacao = round(((atual - inicial) / inicial) * 100, 2)
        sinal_variacao = "+" if variacao >= 0 else ""

        analises = {
            "codigo":         codigo,
            "maxima":         maxima,
            "minima":         minima,
            "atual":          atual,
            "variacao":       variacao,
            "sinal_variacao": sinal_variacao,
            "fechamento":     fechamento
        }

        print(f"📊 Análise da ação {codigo} — últimos {PERIODO_ANALISE}:")
        print(f"   → Cotação Máxima : R$ {maxima}")
        print(f"   → Cotação Mínima : R$ {minima}")
        print(f"   → Cotação Atual  : R$ {atual}")
        print(f"   → Variação       : {sinal_variacao}{variacao}%")

        return analises

    except Exception as e:
        print(f"❌ Erro ao gerar análises: {e}")
        return None


# Gera as análises (somente se os dados foram carregados)
if dados is not None:
    analises = gerar_analises(dados, codigo)
else:
    print("⚠️  Execute a célula anterior para carregar os dados primeiro.")

## 📈 6. Visualizar Histórico de Preços

Gera um gráfico do histórico de fechamento da ação no período analisado.

In [ ]:
def plotar_grafico(analises, periodo):
    """
    Gera um gráfico de linha com o histórico de preços da ação.

    Parâmetros:
        analises (dict): Dicionário com os dados da análise
        periodo  (str) : Período de análise
    """
    try:
        fechamento = analises['fechamento']
        codigo     = analises['codigo']
        cor        = 'green' if analises['variacao'] >= 0 else 'red'

        fig, ax = plt.subplots(figsize=(12, 5))

        ax.plot(fechamento.index, fechamento.values, color=cor, linewidth=2, label=codigo)
        ax.fill_between(fechamento.index, fechamento.values, alpha=0.1, color=cor)

        # Linha horizontal mostrando a cotação atual
        ax.axhline(
            y=analises['atual'],
            color='blue', linestyle='--', linewidth=1, alpha=0.7,
            label=f"Atual: R$ {analises['atual']}"
        )

        ax.set_title(f"Histórico de Fechamento — {codigo} ({periodo})", fontsize=14, fontweight='bold')
        ax.set_xlabel("Data")
        ax.set_ylabel("Preço de Fechamento (R$)")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Formata o eixo X com datas legíveis
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
        fig.autofmt_xdate()

        plt.tight_layout()
        plt.savefig('grafico_acao.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✅ Gráfico gerado e salvo como 'grafico_acao.png'")

    except Exception as e:
        print(f"❌ Erro ao gerar gráfico: {e}")


# Plota o gráfico (somente se as análises foram geradas)
if 'analises' in dir() and analises is not None:
    plotar_grafico(analises, PERIODO_ANALISE)
else:
    print("⚠️  Execute as células anteriores primeiro.")

## 📧 7. Enviar E-mail Automaticamente

Esta etapa usa `pyautogui` para abrir o Gmail no navegador e enviar o e-mail automaticamente.

> ⚠️ **Importante:**
> - Feche outras abas desnecessárias antes de executar
> - Não mova o mouse durante a execução
> - Você terá **5 segundos** após executar a célula para posicionar a janela do navegador

In [ ]:
def montar_mensagem(analises, periodo):
    """
    Monta o corpo do e-mail com as análises formatadas.

    Parâmetros:
        analises (dict): Dicionário com os dados da análise
        periodo  (str) : Período de análise

    Retorna:
        str com o corpo do e-mail formatado.
    """
    codigo         = analises['codigo']
    maxima         = analises['maxima']
    minima         = analises['minima']
    atual          = analises['atual']
    variacao       = analises['variacao']
    sinal_variacao = analises['sinal_variacao']

    mensagem = f"""Prezado(a) Gestor(a),

Seguem as análises diárias da ação {codigo} referentes aos últimos {periodo}:

  Cotação Máxima  : R$ {maxima}
  Cotação Mínima  : R$ {minima}
  Cotação Atual   : R$ {atual}
  Variação Período: {sinal_variacao}{variacao}%

Qualquer dúvida, estou à disposição!

Atenciosamente,
Sistema de Análise Automatizada"""

    return mensagem


def enviar_email_gmail(destinatario, assunto, mensagem):
    """
    Automatiza o envio de e-mail pelo Gmail usando PyAutoGUI.

    Parâmetros:
        destinatario (str): E-mail do destinatário
        assunto      (str): Assunto do e-mail
        mensagem     (str): Corpo do e-mail
    """
    try:
        # Configura pausa entre cada ação do PyAutoGUI
        pyautogui.PAUSE = PYAUTOGUI_PAUSE

        print("⏳ Iniciando envio em 5 segundos... Posicione o navegador!")
        time.sleep(5)

        # Abre uma nova aba no navegador
        print("📌 Abrindo nova aba...")
        pyautogui.hotkey("ctrl", "t")

        # Navega para o Gmail
        print("📌 Acessando o Gmail...")
        pyperclip.copy("www.gmail.com")
        pyautogui.hotkey("ctrl", "v")
        pyautogui.press("enter")

        # Clica no botão "Escrever"
        # ⚠️ As coordenadas (x, y) podem variar conforme sua resolução de tela
        # Para descobrir as coordenadas certas, use: time.sleep(5); print(pyautogui.position())
        print("📌 Clicando em 'Escrever'...")
        pyautogui.click(x=246, y=632)

        # Preenche o destinatário e pressiona Tab para avançar
        print("📌 Preenchendo destinatário...")
        pyperclip.copy(destinatario)
        pyautogui.hotkey("ctrl", "v")
        pyautogui.press("tab")

        # Preenche o assunto e pressiona Tab para avançar
        print("📌 Preenchendo assunto...")
        pyperclip.copy(assunto)
        pyautogui.hotkey("ctrl", "v")
        pyautogui.press("tab")

        # Preenche o corpo do e-mail
        print("📌 Preenchendo corpo do e-mail...")
        pyperclip.copy(mensagem)
        pyautogui.hotkey("ctrl", "v")

        # Envia o e-mail com atalho Ctrl+Enter
        print("📌 Enviando e-mail...")
        pyautogui.hotkey("ctrl", "enter")

        print("✅ E-mail enviado com sucesso!")

    except Exception as e:
        print(f"❌ Erro ao enviar e-mail: {e}")
        print("   Verifique se o navegador está aberto e tente novamente.")


# Envia o e-mail (somente se as análises foram geradas)
if 'analises' in dir() and analises is not None:
    mensagem = montar_mensagem(analises, PERIODO_ANALISE)
    assunto  = f"Análise Diária — {codigo}"

    print("📋 Prévia do e-mail que será enviado:")
    print("-" * 50)
    print(f"Para   : {EMAIL_DESTINATARIO}")
    print(f"Assunto: {assunto}")
    print("-" * 50)
    print(mensagem)
    print("-" * 50)

    confirmar = input("\nDeseja enviar o e-mail? (s/n): ").strip().lower()
    if confirmar == 's':
        enviar_email_gmail(EMAIL_DESTINATARIO, assunto, mensagem)
    else:
        print("🚫 Envio cancelado pelo usuário.")
else:
    print("⚠️  Execute as células anteriores primeiro.")

---

## 🚀 Pipeline Completo (Tudo de Uma Vez)

Execute a célula abaixo para rodar todo o fluxo de uma só vez:  
busca os dados → gera análises → plota gráfico → envia e-mail.

In [ ]:
def pipeline_completo():
    """
    Executa o pipeline completo de análise e envio de e-mail.
    """
    print("=" * 55)
    print("  📈 PIPELINE DE ANÁLISE DE AÇÕES")
    print("=" * 55)

    # 1. Coleta o código da ação
    codigo = input("\nDigite o código da ação (ex: PETR4.SA): ").strip().upper()

    # 2. Busca os dados
    print("\n📦 Etapa 1/4 — Buscando dados...")
    dados = buscar_dados_acao(codigo, PERIODO_ANALISE)
    if dados is None:
        return

    # 3. Gera as análises
    print("\n📊 Etapa 2/4 — Gerando análises...")
    analises = gerar_analises(dados, codigo)
    if analises is None:
        return

    # 4. Plota o gráfico
    print("\n📈 Etapa 3/4 — Gerando gráfico...")
    plotar_grafico(analises, PERIODO_ANALISE)

    # 5. Envia o e-mail
    print("\n📧 Etapa 4/4 — Preparando e-mail...")
    mensagem = montar_mensagem(analises, PERIODO_ANALISE)
    assunto  = f"Análise Diária — {codigo}"

    confirmar = input("Deseja enviar o e-mail? (s/n): ").strip().lower()
    if confirmar == 's':
        enviar_email_gmail(EMAIL_DESTINATARIO, assunto, mensagem)
    else:
        print("🚫 Envio de e-mail cancelado.")

    print("\n" + "=" * 55)
    print("  ✅ PIPELINE CONCLUÍDO COM SUCESSO!")
    print("=" * 55)


# Executa o pipeline completo
pipeline_completo()